In [0]:
business_df = spark.read.json(
    "dbfs:/Workspace/Users/ksaiti@nyc.gr/data7302_yelp_example/data/yelp_academic_dataset_business.json"
)

In [0]:
business_df.printSchema()

In [0]:
business_df.show(5)

In [0]:
from pyspark.sql.functions import col

def flatten_columns(df):

    flat_cols = []

    for field in df.schema.fields:

        field_name = field.name

        if field.dataType.typeName() == "struct":

            nested_cols = [

                col(f"{field_name}.{subfield.name}").alias(
                    f"{field_name}_{subfield.name}"
                )

                for subfield in field.dataType.fields
            ]

            flat_cols.extend(nested_cols)

        else:

            flat_cols.append(col(field_name))

    return df.select(flat_cols)


business_clean = flatten_columns(
    business_df
)

In [0]:
business_clean.printSchema()

In [0]:
business_clean = business_clean.dropna(
    subset=["stars"]
)

In [0]:
from pyspark.sql.functions import when

business_clean = business_clean.withColumn(
    "high_rating",
    when(
        col("stars") >= 4,
        1
    ).otherwise(0)
)

In [0]:
business_clean.createOrReplaceTempView(
    "businesses"
)

In [0]:
spark.sql("""
SELECT
    state,
    COUNT(*) AS businesses
FROM businesses
GROUP BY state
ORDER BY businesses DESC
""").show()


In [0]:
from pyspark.sql.functions import length

business_clean = business_clean.withColumn(
    "name_length",
    length("name")
)

In [0]:
from pyspark.ml.feature import VectorAssembler
from pyspark.ml.classification import LogisticRegression

assembler = VectorAssembler(
    inputCols=[
        "review_count",
        "name_length"
    ],
    outputCol="features"
)

ml_data = assembler.transform(
    business_clean
)

model = LogisticRegression(
    featuresCol="features",
    labelCol="high_rating"
)

model = model.fit(
    ml_data
)

In [0]:
predictions = model.transform(
    ml_data
)

predictions.select(
    "name",
    "stars",
    "prediction"
).show()

In [0]:
display(
    spark.sql("""
    SELECT
        state,
        AVG(stars) AS avg_rating
    FROM businesses
    GROUP BY state
    """)
)